In [1]:
using DelimitedFiles
using Random
using LinearAlgebra

In [2]:
const Np::Int64 = 50 #number of positive vortices
const Nm::Int64 = 50 #number of negative vortices
N = Np + Nm
const Lx::Float64 = 2.0 * pi
const Ly::Float64 = 2.0 * pi 

dt = 5.e-2
dtMax = 5.e-2
xy = zeros(N,2)
g = zeros(N,1)
const totalSteps::Int64 = 1000000
const outputTime::Float64 = 0.1
fileNumber = 0
timeCurrent = 0.0
const rkTolerance::Float64 = 1.e-6
const imageTotal::Int64 = 5

const FLAG_Boundary::String = "periodic" #periodic, periodic-y, infinite
const FLAG_BackgroundFlow::Bool = false
shearStrength = 1.0

1.0

In [3]:
function initialise!(xy,g)

    timeCurrent, fileNumber = readdlm("./data/curframe.dat")
    fileNumber = Int64(fileNumber)
    
    if fileNumber < 0
     
        for index in 1:Np
            xy[index,:] = [0,0.1]#[Lx*rand(),Ly*rand()]
            g[index,1] = 1.0/N 
        end
        for index in Np+1:N
            xy[index,:] = [0,-0.1]#[Lx*rand(),Ly*rand()] 
            g[index,1] = -1.0/N 
        end
        timeCurrent = 0.0
        fileNumber = 0
        writedlm( "./data/vortex_xyg.$(lpad(fileNumber,5,"0"))",  [xy g], '\t')
        writedlm( "./data/curframe.dat", zip(timeCurrent, fileNumber), '\t')
    else  
        dataIn =  readdlm( "./data/vortex_xyg.$(lpad(fileNumber,5,"0"))")
        copyto!(xy,dataIn[:,1:2])
        copyto!(g,dataIn[:,3])
      
    end
    
    return (timeCurrent, fileNumber)
end

initialise! (generic function with 1 method)

In [4]:
function computeVelocity(xy)
    dv = zeros(N,2)

    if FLAG_BackgroundFlow == true
        
        for iIndex in 1:N
            dv[iIndex,1] += shearStrength * (xy[iIndex,1]-0.5*Lx)
        end
    end
    
    if FLAG_Boundary == "infinite"
        for iIndex in 1:N
            for jIndex in 1:N
                if jIndex != iIndex
                    xij = xy[iIndex,1]- xy[jIndex,1]
                    yij = xy[iIndex,2]- xy[jIndex,2]
                    r2ij = xij^2 + yij^2
                    dv[iIndex,:] += [-(0.5/pi)* g[jIndex,1] * yij/ r2ij, (0.5/pi)* g[jIndex,1] * xij/ r2ij]
                end
            end
        end
        return dv

    elseif FLAG_Boundary == "periodic"
        for iIndex in 1:N
            for jIndex in 1:N
                if jIndex != iIndex
                    xij = xy[iIndex,1]- xy[jIndex,1]
                    yij = xy[iIndex,2]- xy[jIndex,2]
                    
                    dv[iIndex,:] += [-(0.25/pi)* g[jIndex,1] * sin(yij)/ (cosh(xij)-cos(yij)), (0.25/pi)* g[jIndex,1] * sin(xij)/ (cosh(yij)-cos(xij))]

                    for nImage in 1:imageTotal
                        dv[iIndex,:] += [-(0.25/pi)* g[jIndex,1] * sin(yij)/ (cosh(xij-2.0*pi*nImage)-cos(yij))-(0.25/pi)* g[jIndex,1] * sin(yij)/ (cosh(xij+2.0*pi*nImage)-cos(yij)), (0.25/pi)* g[jIndex,1] * sin(xij)/ (cosh(yij-2.0*pi*nImage)-cos(xij))+(0.25/pi)* g[jIndex,1] * sin(xij)/ (cosh(yij+2.0*pi*nImage)-cos(xij))]
                    end
                end
            end
        end
        return dv
    elseif FLAG_Boundary == "periodic-y"
        for iIndex in 1:N
            for jIndex in 1:N
                if jIndex != iIndex
                    xij = xy[iIndex,1] - xy[jIndex,1]
                    yij = xy[iIndex,2] - xy[jIndex,2]
                    
                    dv[iIndex,:] += [-(0.25/pi) * g[jIndex,1] * sin(yij) / (cosh(xij)-cos(yij)), (0.25/pi)* g[jIndex,1] * sin(xij)/ (cosh(yij)-cos(xij))]
                end
            end
        end
        return dv

    end
end

computeVelocity (generic function with 1 method)

In [5]:
function invokeBoundaryConditions!(xy)
    if FLAG_Boundary == "infinite"
        return nothing
    elseif FLAG_Boundary == "periodic"
        for iIndex = 1:N
            if xy[iIndex,1] >= Lx
                xy[iIndex,1] -= Lx
            elseif xy[iIndex,1] < 0.0
                xy[iIndex,1] += Lx
            end
        
            if xy[iIndex,2] >= Ly
                xy[iIndex,2] -= Ly
            elseif xy[iIndex,2] < 0.0
                xy[iIndex,2] += Ly
            end
        end
        return nothing
    elseif FLAG_Boundary == "periodic-y"
        for iIndex = 1:N
            if xy[iIndex,2] >= Ly
                xy[iIndex,2] -= Ly
            elseif xy[iIndex,2] < 0.0
                xy[iIndex,2] += Ly
            end  
        end
        return nothing
    end

end

invokeBoundaryConditions! (generic function with 1 method)

In [6]:
function rungeKutta45(xy,dt)

    k1 = dt * computeVelocity(xy)
    k2 = dt * computeVelocity(xy + (k1/5.0))
    k3 = dt * computeVelocity(xy + (3.0/40.0)*k1 + (9.0/40.0)*k2 )
    k4 = dt * computeVelocity(xy + (3.0/10.0)*k1 - (9.0/10.0)*k2 + (6.0/5.0)*k3)
    k5 = dt * computeVelocity(xy - (11.0/54.0)*k1 + (5.0/2.0)*k2 - (70.0/27.0)*k3 + (35.0/27.0)*k4)
    k6 = dt * computeVelocity(xy + (1631.0/55296.0)*k1 + (175.0/512.0)*k2 + (575.0/13824.0)*k3 + (44275.0/110592.0)*k4 + (253.0/4096.0)*k5)

    sol4 = xy + (37.0/378.0)*k1 + (250.0/612.0)*k3 + (125.0/594.0)*k4 + (512.0/1771.0)*k6;
    sol5 = xy + (2825.0/27648.0)*k1 + (18575.0/48384.0)*k3 + (13525.0/55296.0)*k4 + (277.0/14336.0)*k5 + (1.0/4.0)*k6;


    #adaptive timestep
  errorSol = maximum(maximum.(abs.(sol4 - sol5))) 
  Safety = 0.9
    if errorSol > 1.e12
	    println("Error in timestepping routine is too large....code aborted")
	    exit(code=1)
    elseif errorSol < rkTolerance
	    dtNew = min( Safety * dt * abs(rkTolerance/errorSol)^0.25, 5.0*dt, dtMax)
	    return sol5, dtNew;
    else
      dtNew = Safety * dt * abs(rkTolerance/errorSol)^0.2
	    xy, dt = rungeKutta45(xy,max(dtNew, 0.1*dt));
	    return xy, dt
    end

end

rungeKutta45 (generic function with 1 method)

In [7]:
function hamiltonian(xy,g)
    hamiltonianValue = 0.0
    momentumXValue = 0.0
    momentumYValue = 0.0
    
    momentumXValue = dot(g,xy[:,1])
    momentumYValue = dot(g,xy[:,2])
    
    if FLAG_Boundary == "periodic"
        for iIndex in 1:N
            for jIndex in 1:N
                if jIndex > iIndex
                    xij = xy[iIndex,1]- xy[jIndex,1]
                    yij = xy[iIndex,2]- xy[jIndex,2]
                   
                    hamiltonianValue -= (0.5/pi) * g[iIndex,1] * g[jIndex,1] * log(cosh(xij)-cos(yij))

                    for nImage in 1:imageTotal
                        hamiltonianValue -= (0.5/pi) * g[iIndex,1] * g[jIndex,1] * log(1.0 + (sinh(xij)*sinh(xij)*(1.0 - tanh(2.0*pi*nImage)*tanh(2.0*pi*nImage)) ) + (cos(yij)*cos(yij)/(cosh(2.0*pi*nImage)*cosh(2.0*pi*nImage))) - (2.0*cosh(xij)*cos(yij)/cosh(2.0*pi*nImage)))
                    end

                    hamiltonianValue += (0.25/pi^2) * g[iIndex,1] * g[jIndex,1] *xij^2
                                 

                end
            end
        end
    elseif FLAG_Boundary == "periodic-y"
        for iIndex in 1:N
            for jIndex in 1:N
                if jIndex > iIndex
                    xij = xy[iIndex,1]- xy[jIndex,1]
                    yij = xy[iIndex,2]- xy[jIndex,2]
                   
                    hamiltonianValue -= (0.5/pi) * g[iIndex,1] * g[jIndex,1] * log(cosh(xij)-cos(yij))

                    hamiltonianValue += (0.25/pi^2) * g[iIndex,1] * g[jIndex,1] * xij^2
                                 
                end
            end
        end

    elseif FLAG_Boundary == "infinite"
        for iIndex in 1:N
            for jIndex in 1:N
                if jIndex > iIndex
                    xij = xy[iIndex,1]- xy[jIndex,1]
                    yij = xy[iIndex,2]- xy[jIndex,2]
                    rij = sqrt(xij^2 + yij^2)
                    hamiltonianValue -= (0.5/pi) * g[iIndex,1] * g[jIndex,1] * log(rij)
                end
            end
        end
    end
    writedlm( "./data/hamiltonian.$(lpad(fileNumber,5,"0"))",  [timeCurrent hamiltonianValue momentumXValue momentumYValue], '\t')
    println([timeCurrent hamiltonianValue momentumXValue momentumYValue])

end

hamiltonian (generic function with 1 method)

In [8]:
#initialise
timeCurrent, fileNumber = initialise!(xy,g)

for stepNumber in 1:totalSteps
   
    xy, dt = rungeKutta45(xy,dt)
    
    invokeBoundaryConditions!(xy)


    timeCurrent += dt
   
    if Int64(div(timeCurrent,outputTime)) == fileNumber +1  
        fileNumber +=  1 
        hamiltonian(xy,g)
        println( "fileNumber = ", fileNumber," time = ", timeCurrent, " dt = ", dt,  '\n')
        writedlm( "./data/vortex_xyg.$(lpad(fileNumber,5,"0"))",  [xy g], '\t')
        writedlm( "./data/curframe.dat", zip(timeCurrent, fileNumber), '\t')
    end
end





ArgumentError: ArgumentError: Cannot open './data/vortex_xyg.00000': not a file

In [9]:
d=0.1
x12=0
y12=2*d
r12 = sqrt(x12^2 + y12^2)
ham = 0
vel1x = 0
ham += (0.5/pi)*log( r12 )
vel1x += (0.5/pi)*y12/ (r12^2) 
println("single dipole: vel1x = ", vel1x, " ham = ", ham)

for n in  1:100000000
    vel1x +=  (0.5/pi)*( - (1/(2*pi*n)) + (1/( 2*d + 2*pi*n)))
    vel1x +=  (0.5/pi)*(  (1/(2*pi*n)) + 1/(2*d-2*pi*n))
    ham += (0.5/pi)*( log( sqrt(x12^2 + (y12 + 2*pi*n)^2) ) - log( abs(2*pi*n) ) + log( sqrt(x12^2 +(y12 -2*pi*n)^2))- log( abs(-2*pi*n)))
end

println("periodic-y dipole from infinite : vel1x = ", vel1x, " ham = ", ham)

vel1x =0
ham = 0
vel1x = (0.25/pi)*sin(y12)/(cosh(x12) - cos(y12))
ham = (0.25/pi) * log(cosh(x12)-cos(y12)) 
ham -= (0.125/(pi^2))*x12^2
println("periodic-y dipole: vel1x = ", vel1x, " ham = ", ham)